# Train Machine Learning Models

## Overview
- Test a range of different models.
- Test a range of different hyperparameters.
- Test a range of different inputs. 

In [1]:
# Import libraries.
import os
import random
import json
import pandas as pd
import ast
import numpy as np
from glob import glob
from collections import Counter

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, make_scorer, confusion_matrix
)

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier

import pandas as pd
import numpy as np
import random
import torch
import re
import nltk
from nltk.corpus import stopwords

import shap
from itertools import product


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix


from sklearn.feature_extraction.text import TfidfVectorizer

import numpy as np
import pandas as pd
import os
import random
from sklearn.preprocessing import StandardScaler

from scipy import sparse

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

N_SPLITS_CV = 5
N_JOBS = 1  # prioritizing no leakage

In [3]:
# Make stop words.
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))
stop_words.extend(['mrs', 'ms', 'mr', 'am', 'pm'])
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}
# Load the dataset
dataset = pd.read_csv("./dataSyntheticAll.csv")
real_dataset_for_labelling = pd.read_csv("./dataRealAll.csv")
dataset["label"] = dataset["needs"].map(label_map)

# Make function to remove punctuation, make lowercase, remove names, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        # lowercase + extract words only
        tokens = re.findall(r"\b[a-zA-Z]+\b", note.lower())
        
        # filter stopwords
        tokens = [t for t in tokens if t not in stop_words]
        
        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

dataset['report'] = preprocessing(dataset['report'].values.tolist())
real_dataset_for_labelling['Note'] = preprocessing(real_dataset_for_labelling['Note'].values.tolist())
real_samples = real_dataset_for_labelling['Note'].values.tolist()

# Split the dataset. (80/10/10)
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=dataset["label"]
)
y_train = train_df['label'].values
y_test = test_df['label'].values

# Check distributions.
def check_distribution(dataframe, name):
    counts = dataframe["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_df, "Train")
check_distribution(test_df, "Test")

[nltk_data] Downloading package stopwords to /home/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Test distribution:
label
0    0.502161
1    0.497839
Name: proportion, dtype: float64


In [4]:
# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # fast and strong
# train_embeddings = embedding_model.encode(train_df['report'].tolist(), show_progress_bar=True)
# test_embeddings  = embedding_model.encode(test_df['report'].tolist(), show_progress_bar=True)

# tfidf and reduction parameters
tfidf_params = {
    "max_features": [10000, 20000],
    "ngram_range": [(1,1), (1,2)],
    "min_df": [2,5]
}

# reduction_methods = {
#     "pca": PCA,
#     "ica": FastICA,
#     "rp": GaussianRandomProjection,
#     "fa": FactorAnalysis,
#     "umap": umap.UMAP
# }

# inputs_dict = {'embeddings': train_embeddings}
# test_dict = {'embeddings': test_embeddings}


inputs_dict = {}
test_dict = {}
real_dict = {}


# tf-idf representations
for max_feat, n_gram, min_df in product(tfidf_params['max_features'],
                                       tfidf_params['ngram_range'],
                                       tfidf_params['min_df']):
    key = f"tfidf_{max_feat}_{max(list(n_gram))}_{min_df}"
    vectorizer = TfidfVectorizer(max_features=max_feat, ngram_range=n_gram, min_df=min_df)
    inputs_dict[key] = vectorizer.fit_transform(train_df['report'].tolist())
    test_dict[key] = vectorizer.transform(test_df['report'].tolist())
    real_dict[key] = vectorizer.transform(real_samples)

In [5]:
# get models - broad coverage across linear, margin, instance-based, and nonlinear tree ensembles
MODELS = {
    # high numbers of iterations (max_iter) avoids convergence warnings in high-dimensional spaces
    "LogisticRegression": LogisticRegression(max_iter=20000, class_weight="balanced", random_state=RANDOM_STATE), # strong baseline
    "RidgeClassifier": RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE), # strong baseline
    "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced", random_state=RANDOM_STATE), # strong baseline
    "SVC-RBF": SVC(probability=False, random_state=RANDOM_STATE, class_weight="balanced"), # probability set to false saves on compute
    "SGD-Hinge": SGDClassifier(loss="hinge", max_iter=20000, random_state=RANDOM_STATE, class_weight="balanced"), # hinge loss is good for binary classification problems - fast on large data
    "KNN": KNeighborsClassifier(), # deterministic - sensitive to scaling (good to test with and without standard scaler)
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"), # don't need scaling and captures nonlinear splits
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"), # don't need scaling and captures nonlinear splits
    "ExtraTrees": ExtraTreesClassifier(random_state=RANDOM_STATE, class_weight="balanced"), # don't need scaling and captures nonlinear splits
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE), # don't need scaling and captures nonlinear splits
    "HistGB": HistGradientBoostingClassifier(random_state=RANDOM_STATE), # don't need scaling and captures nonlinear splits
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE), # don't need scaling and captures nonlinear splits
    "GaussianNaiveBayes": GaussianNB(), # deterministic - interpretable
}


# make pipeline with and without scaling
def preprocessor(scale):
    # median ensures it is robust to outliers
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        # scales data to benefit models such as SVM and KNN
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        steps.append(("scaler", FunctionTransformer(lambda X: X, feature_names_out="one-to-one")))
    return Pipeline(steps)


def gmean_binary_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0   # sensitivity
    tnr = tn / (tn + fp) if (tn + fp) else 0.0   # specificity
    return np.sqrt(tpr * tnr)

gmean_scorer = make_scorer(gmean_binary_score)

def build_scorers():
    return {
        "accuracy": "accuracy", # normal accuracy
        "balanced_accuracy": make_scorer(balanced_accuracy_score), # balanced accuracy
        "precision": make_scorer(precision_score, average="binary", zero_division=0),
        "recall": make_scorer(recall_score, average="binary", zero_division=0),
        "f1": make_scorer(f1_score, average="binary", zero_division=0),
        "roc_auc": "roc_auc",
        "gmean": gmean_scorer
    }

In [6]:
final_results = pd.read_csv('./baseline_model_results.csv')

metrics = {
    "Accuracy": "holdout_accuracy",
    "Balanced Accuracy": "holdout_bal_acc",
    "Precision": "holdout_precision",
    "Recall": "holdout_recall",
    "F1": "holdout_f1",
    "ROC AUC": "holdout_roc_auc",
    "Geometric Mean": "holdout_gmean"
}

best_models = []
different_feature_sets = []
different_variants = []
for name, col in metrics.items():
    temp_values = final_results.sort_values(by=col, ascending=False)
    model_name = temp_values['model'].tolist()[0]
    clf = MODELS[model_name]
    params = json.loads(temp_values['clf_parameters'].tolist()[0])
    
    tuned_params = json.loads(temp_values['tuned_parameters'].tolist()[0])
    for param, value in tuned_params.items():
        params[param.replace('clf__', '')] = value
    clf.set_params(**params)
    best_models.append(clf)
    different_feature_sets.append(temp_values['feature_set'].tolist()[0])
    different_variants.append(temp_values['variant'].tolist()[0])
    print(f"Highest {col} is {model_name} model with {params}, the {temp_values['feature_set'].tolist()[0]} feature set and {temp_values['variant'].tolist()[0]} variant.")

X_train = inputs_dict[Counter(different_feature_sets).most_common(1)[0][0]]
X_test = test_dict[Counter(different_feature_sets).most_common(1)[0][0]]
real_data_for_labelling = real_dict[Counter(different_feature_sets).most_common(1)[0][0]]
variant = Counter(different_variants).most_common(1)[0][0]

Highest holdout_accuracy is LinearSVC model with {'C': 0.5, 'class_weight': 'balanced', 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 20000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 1618, 'tol': 0.0001, 'verbose': 0}, the tfidf_20000_2_2 feature set and raw variant.
Highest holdout_bal_acc is LinearSVC model with {'C': 0.5, 'class_weight': 'balanced', 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 20000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 1618, 'tol': 0.0001, 'verbose': 0}, the tfidf_20000_2_2 feature set and raw variant.
Highest holdout_precision is LogisticRegression model with {'C': 0.1, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 20000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 1618, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}, th

In [7]:
scorers = build_scorers()
results = []

# scaling decision
X_tr, X_te = X_train, X_test


ensemble = VotingClassifier(
    estimators=[(f"model_{i}", m) for i, m in enumerate(best_models)],
    voting="hard" # some models do not have predict_proba
)


# Scaling decision
if variant == "scaled":
    scaler = StandardScaler(with_mean=False if sparse.issparse(X_tr) else True)
else:
    scaler = FunctionTransformer(lambda X: X, feature_names_out="one-to-one")

X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

ensemble.fit(X_tr, y_train)

y_pred = ensemble.predict(X_te)


# no roc_auc because some models do not have predict_proba
# y_score = None
# if hasattr(ensemble, "predict_proba"):
#     y_score = ensemble.predict_proba(X_te)[:, 1]


accuracy = accuracy_score(y_test, y_pred)
bal_accuracy = balanced_accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
# roc_auc = roc_auc_score(y_test, y_score) if y_score is not None else np.nan
g_mean_score = gmean_binary_score(y_test, y_pred)


In [8]:
accuracy, bal_accuracy, precision, recall, f1, g_mean_score

(0.9317199654278306,
 0.9318494214955059,
 0.9067103109656302,
 0.9618055555555556,
 0.9334456613310868,
 np.float64(0.9313677975825047))

In [9]:
real_pred = ensemble.predict(real_data_for_labelling)

In [10]:
# real_dataset_for_labelling['Resident Study Number'].values.tolist()
real_dataset_for_labelling

,Resident Study Number,Date,Time,Note Type,Note,Pain Indicated,Nausea Indicated,SPC Referral Indicated,Delirium Indicated
0,P1,2025-01-06,01:54:00,Night note,assisted personal care needs required due meds...,NaN,NaN,NaN,NaN
1,P1,2025-01-06,12:21:00,Priority entry,holistic care plan resident care plan evaluati...,NaN,NaN,NaN,NaN
2,P1,2025-01-06,13:00:00,Day note,resident appears good form due meds given char...,NaN,NaN,NaN,NaN
3,P1,2025-01-07,01:30:00,Night note,resident remains pleasant upbeat self caring p...,NaN,NaN,NaN,NaN
4,P1,2025-01-07,09:01:00,Other,type blood details full bloods including e iro...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
11816,P9,2025-08-05,13:16:00,Day note,resident appears bright alert due medications ...,NaN,NaN,NaN,NaN
11817,P9,2025-08-06,01:54:00,Night note,resident settled night meds charted compliant ...,NaN,NaN,NaN,NaN
11818,P9,2025-08-06,15:20:00,Day note,resident pleasantly confused compliant dressin...,NaN,NaN,NaN,NaN
11819,P9,2025-08-06,18:37:00,Day note,influenza vaccination given influvac batch bat...,NaN,NaN,NaN,NaN


In [11]:
real_data_label_df = pd.DataFrame({'id': real_dataset_for_labelling['Resident Study Number'].values.tolist(), 'note': real_samples, 'label': real_pred}) # fill nan id values with 'P15' as this is lost in the 
real_data_label_df_shuffled = real_data_label_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
# Save labels for all real notes.
real_data_label_df.to_csv('./met_unmet_labelled_real_notes.csv')
# Get top 5 shuffled notes under each label.
real_met = real_data_label_df_shuffled[real_data_label_df_shuffled['label'] == 0].iloc()[:5]
real_unmet = real_data_label_df_shuffled[real_data_label_df_shuffled['label'] == 1].iloc()[:5]
# Save sample notes for synthetic generation. 
(pd.DataFrame({'Note': real_unmet['note'].tolist() + real_met['note'].tolist(), 'Needs': (['unmet'] * 5) + (['met'] * 5)})).to_excel('./real_notes.xlsx')
